Fetch form definitions


In [ ]:
import {
  REGISTRAR_USERNAME,
  REGISTRAR_PASSWORD
} from '../commonHelpers/vars.ts'
import { authenticate } from '../commonHelpers/authentication.ts'

const token = await authenticate(REGISTRAR_USERNAME, REGISTRAR_PASSWORD)
token

In [ ]:
import { extractFieldType, Field } from '../commonHelpers/utils.ts'
import { fetchEvents } from '../commonHelpers/formsHandlers.ts'

const ignoredFields = [
  'DIVIDER',
  'PARAGRAPH',
  'BULLET_LIST',
  'HEADING3',
  'SIGNATURE',
  'PAGE_HEADER',
  'FILE',
  'FILE_WITH_OPTIONS',
  'SEARCH',
  'VERIFICATION_STATUS',
  'QUERY_PARAM_READER',
  'HTTP',
  'LOADER',
  'ID_READER',
  'collector.*',
  'requester.*',
  'review.*',
  'fees.*',
  'lateFee.*',
  'documents.*',
  'informant.contact',
  'reason.option',
  'reason.other'
]

const events = await fetchEvents(token)

type StreetAddressField = { id: string; type: string; addressType: 'DOMESTIC' | 'INTERNATIONAL' }
type EventFieldDescriptor = {
  id: string
  type: string
  options?: string[]
  nameFields?: string[]
  streetAddressFields?: StreetAddressField[]
  defaultCountry?: string
  asOfDateRef?: string
}

const eventDescription = events.reduce(
  (allEvents, event) => {
    const uniqueFields = new Map<string, EventFieldDescriptor>()

    extractFieldType(event, 'fields')
      .filter((x) => !ignoredFields.includes(x.type))
      .filter((x) => x.id)
      .filter(
        (x) => !ignoredFields.some((pattern) => new RegExp(pattern).test(x.id))
      )
      .forEach((f) => {
        uniqueFields.set(f.id, {
          id: f.id,
          type: f.type,
          options: f.options?.map((o) => o.value),
          ...(f.type === 'NAME' && f.configuration?.name
            ? { nameFields: Object.keys(f.configuration.name as Record<string, unknown>) }
            : {}),
          ...(f.type === 'ADDRESS' && f.configuration?.streetAddressForm
            ? {
                streetAddressFields: (f.configuration.streetAddressForm as { id: string; type: string; conditionals?: unknown }[]).map((s) => ({
                  id: s.id,
                  type: s.type,
                  addressType: JSON.stringify(s.conditionals).includes('DOMESTIC') ? 'DOMESTIC' : 'INTERNATIONAL'
                })),
                ...((f.defaultValue as Record<string, unknown>)?.country
                  ? { defaultCountry: (f.defaultValue as Record<string, unknown>).country as string }
                  : {})
              }
            : {}),
          ...(f.type === 'COUNTRY' && typeof f.defaultValue === 'string'
            ? { defaultCountry: f.defaultValue }
            : {}),
          ...(f.type === 'AGE' && (f.configuration as Record<string, unknown>)?.asOfDate
            ? { asOfDateRef: ((f.configuration as Record<string, unknown>).asOfDate as Record<string, unknown>)['$$field'] as string }
            : {})
        })
      })

    allEvents[event.id] = [...uniqueFields.values()]

    return allEvents
  },
  {} as Record<string, EventFieldDescriptor[]>
)

Deno.writeFileSync(
  './formData/eventDescription.json',
  new TextEncoder().encode(JSON.stringify(eventDescription, null, 2))
)

Deno.writeFileSync(
  './formData/fullEvent.json',
  new TextEncoder().encode(JSON.stringify(events, null, 2))
)

In [ ]:
import { generateTypes } from './helpers/typeGenerator.ts'

generateTypes(eventDescription)